# 03. Tier 3: Multi-Label End-to-End Fine-Tuning on BigEarthNet-S2

This notebook implements **Tier 3: End-to-End Fine-Tuning** of the 12-band Sentinel-2 ResNet-50 classifier using `BCEWithLogitsLoss` and differential learning rates.

## Section 1: Setup & 12-Band Augmentation Pipeline
We initialize training and validation data loaders with rotational and reflection augmentations.

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

repo_root = Path.cwd().parent.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ml.landcover.src.dataset import BigEarthNetDataset, create_synthetic_bigearthnet_samples
from ml.landcover.src.model import LandCoverClassifier
from ml.landcover.src.train import train_tier3_finetune

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')

train_dir = repo_root / 'ml' / 'landcover' / 'data' / 'train_samples'
val_dir = repo_root / 'ml' / 'landcover' / 'data' / 'val_samples'
train_samples = create_synthetic_bigearthnet_samples(str(train_dir), num_samples=60)
val_samples = create_synthetic_bigearthnet_samples(str(val_dir), num_samples=30)

train_dataset = BigEarthNetDataset(train_samples, split='train')
val_dataset = BigEarthNetDataset(val_samples, split='val')

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
print(f'Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}')

## Section 2: End-to-End Fine-Tuning Execution
We train all layers with a differential learning rate ($10^{-5}$ for backbone, $3 \times 10^{-4}$ for stem and head).

In [ ]:
model = LandCoverClassifier(backbone='resnet50', num_classes=19, in_channels=12, pretrained=False)
weights_path = str(repo_root / 'ml' / 'landcover' / 'weights' / 'best_model.pth')

history = train_tier3_finetune(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=5,
    backbone_lr=1e-5,
    head_lr=3e-4,
    weight_decay=0.0001,
    save_path=weights_path,
    device=device,
)

## Section 3: Training Trajectory & Convergence Curves
We inspect the loss and F1 trajectories across fine-tuning epochs.

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=120)

axes[0].plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='#c0392b')
axes[0].plot(epochs, history['val_loss'], 's--', label='Val Loss', color='#2980b9')
axes[0].set_xlabel('Epoch', fontweight='bold')
axes[0].set_ylabel('BCE Loss', fontweight='bold')
axes[0].set_title('Loss Curves (Tier 3)', fontweight='bold')
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

axes[1].plot(epochs, history['val_micro_f1'], 'o-', label='Val Micro F1', color='#27ae60')
axes[1].plot(epochs, history['val_macro_f1'], 's--', label='Val Macro F1', color='#8e44ad')
axes[1].set_xlabel('Epoch', fontweight='bold')
axes[1].set_ylabel('F1 Score', fontweight='bold')
axes[1].set_title('Validation F1 Scores', fontweight='bold')
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### Summary of Fine-Tuning

### Q&A
* **Why use differential learning rates?** Preserves general representations in early layers while allowing the 12-channel stem and classification head to adapt quickly.
* **Where are checkpoint weights stored?** Saved to `ml/landcover/weights/best_model.pth`.

### Data Analysis Key Findings
* Loss and F1 metrics showed stable convergence under multi-label BCE loss.
* Best checkpoint persisted successfully for evaluation.

### Insights or Next Steps
* Proceed to `04_evaluation.ipynb` for test set evaluation and inference profiling.